# TP 3 — Maladie cardiaque : réseaux de neurones et comparaison de modèles (correction)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/ai_for_medicine/blob/main/ia-medecine/notebooks/iam-s3-tp-correction.ipynb)

*IA pour la médecine — séance 3.*

## Objectif

Ce dernier TP réunit tout ce qui a été vu aux séances 1 et 2 :

1. appliquer la **démarche de préparation des données** (séance 2) à un nouveau jeu de données ;
2. entraîner et comparer **trois modèles** : régression logistique, forêt aléatoire et
   **réseau de neurones** (perceptron multicouche, *MLP*) ;
3. comprendre comment un réseau de neurones apprend et comment le régler.

### Le jeu de données : maladie cardiaque (Statlog)

270 patients, 13 variables cliniques. La tâche : prédire la **présence ou l'absence
d'une maladie cardiaque**. Certaines variables sont vraiment numériques (âge, pression
artérielle), d'autres sont des entiers qui codent des catégories (type de douleur
thoracique, thalassémie) : c'est fréquent dans les données médicales réelles.

### La démarche

Toujours la même : **explorer → visualiser → préparer → entraîner → évaluer → comparer**.

**Comment travailler** : exécutez les cellules dans l'ordre (Maj + Entrée) et complétez
les cellules marquées `# TODO`.

---
## 1. Préparation de l'environnement

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, recall_score, ConfusionMatrixDisplay,
                             classification_report)

import warnings
warnings.filterwarnings("ignore")  # masque les avertissements de convergence du MLP

---
## 2. Charger et découvrir les données

**13 variables :**

| Variable | Signification |
|---|---|
| `age` | âge (années) |
| `sexe` | 1 = homme, 0 = femme |
| `type_douleur_thoracique` | 1 angor typique, 2 angor atypique, 3 douleur non angineuse, 4 asymptomatique |
| `pression_arterielle_repos` | pression artérielle au repos (mmHg) |
| `cholesterol` | cholestérol sérique (mg/dL) |
| `glycemie_jeun_sup_120` | glycémie à jeun > 120 mg/dL (1 = oui) |
| `ecg_repos` | ECG au repos : 0 normal, 1 anomalie ST-T, 2 hypertrophie ventriculaire gauche |
| `frequence_cardiaque_max` | fréquence cardiaque maximale atteinte à l'effort |
| `angor_effort` | angor provoqué par l'effort (1 = oui) |
| `depression_st` | sous-décalage du segment ST à l'effort (mm) |
| `pente_st` | pente du segment ST à l'effort : 1 ascendante, 2 plate, 3 descendante |
| `nb_vaisseaux_colores` | nombre de gros vaisseaux visibles en fluoroscopie (0 à 3) |
| `thalassemie` | scintigraphie au thallium : 3 normal, 6 défaut fixe, 7 défaut réversible |

**Cible :** `maladie_cardiaque`, 1 = présente, 0 = absente.

*Source : Statlog (Heart), UCI Machine Learning Repository, CC BY 4.0.*

### Exercice 1.1 — Charger les données

Lisez le fichier avec `pd.read_csv(URL)` dans `df`, affichez sa taille et ses
premières lignes.

In [ ]:
URL = "https://raw.githubusercontent.com/racousin/ai_for_medicine/main/ia-medecine/donnees/maladie_cardiaque.csv"
df = pd.read_csv(URL)
print("Taille :", df.shape)
df.head()

### Exercice 1.2 — Statistiques et types

Affichez les types (`df.info()`) et les statistiques descriptives (`df.describe()`).

**Question :** quelles variables représentent des catégories, bien qu'elles soient
stockées sous forme de nombres ?

In [ ]:
df.info()
df.describe()

**Réponse.** `sexe`, `type_douleur_thoracique`, `glycemie_jeun_sup_120`, `ecg_repos`,
`angor_effort`, `pente_st`, `thalassemie` sont des codes de catégories
(`nb_vaisseaux_colores` est un comptage, ordonné). Leur moyenne n'a pas de sens
clinique : « type de douleur moyen = 3,2 » ne veut rien dire.

### Exercice 1.3 — Équilibre des classes

Comptez les patients malades et non malades.

**Pourquoi :** comme à la séance 1, pour savoir si l'exactitude est une métrique fiable.

*Indice :* `y.value_counts(normalize=True)`

In [ ]:
y = df["maladie_cardiaque"]
print(y.value_counts())
print(y.value_counts(normalize=True))

---
## 3. Visualiser les données

### Exercice 2.1 — Distributions selon la classe

Tracez la distribution de `age` et de `frequence_cardiaque_max`, colorée par classe.

*Indice :* `sns.histplot(data=df, x="age", hue="maladie_cardiaque", kde=True)`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x="age", hue="maladie_cardiaque", kde=True, ax=axes[0], bins=20)
axes[0].set_title("Âge selon la classe")
sns.histplot(data=df, x="frequence_cardiaque_max", hue="maladie_cardiaque", kde=True,
             ax=axes[1], bins=20)
axes[1].set_title("Fréquence cardiaque maximale selon la classe")
plt.tight_layout()
plt.show()

### Exercice 2.2 — Nuage de points

Tracez `age` en fonction de `frequence_cardiaque_max`, coloré par classe : les deux
variables ensemble séparent-elles les classes ?

*Indice :* `sns.scatterplot(data=df, x="age", y="frequence_cardiaque_max", hue="maladie_cardiaque", alpha=0.6)`

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="age", y="frequence_cardiaque_max", hue="maladie_cardiaque",
                alpha=0.6)
plt.title("Âge et fréquence cardiaque maximale")
plt.show()

### Exercice 2.3 — Carte des corrélations

Affichez la matrice des corrélations de toutes les variables.

*Indice :* `sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)`

**Question :** quelles variables semblent les plus liées à la maladie ?

In [ ]:
plt.figure(figsize=(14, 12))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Corrélations")
plt.tight_layout()
plt.show()

print(df.corr()["maladie_cardiaque"].drop("maladie_cardiaque").abs()
      .sort_values(ascending=False))

**Réponse.** `thalassemie`, `nb_vaisseaux_colores`, `type_douleur_thoracique`, `angor_effort`,
`depression_st` et `frequence_cardiaque_max` (corrélation négative : une fréquence
maximale basse va avec la maladie) sont les plus liées à la cible. Une corrélation ne
mesure qu'un lien linéaire, et sur des codes de catégories elle n'est qu'indicative.

---
## 4. Préparer les données

Ce jeu n'a pas de valeurs manquantes et toutes les variables sont déjà numériques :
il suffit de **standardiser**. Pour les arbres, l'échelle ne compte pas ; pour la
régression logistique et surtout pour les réseaux de neurones, elle est importante.

### Exercice 3.1 — Découper et standardiser

Séparez `X` (toutes les colonnes sauf la cible) et `y`, découpez 80 % / 20 %
(`random_state=42`), puis standardisez avec un `StandardScaler` **ajusté sur
l'entraînement seulement**.

In [ ]:
X = df.drop(columns="maladie_cardiaque")
y = df["maladie_cardiaque"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Entraînement : {X_train.shape[0]} patients, {X_train.shape[1]} variables")
print(f"Test : {X_test.shape[0]} patients")

---
## 5. Modèle de référence : la régression logistique

On commence toujours par un modèle de référence simple.

### Exercice 4.1 — Régression logistique

Entraînez `LogisticRegression(max_iter=5000, random_state=42)` et affichez
l'exactitude, le rapport de classification et la matrice de confusion.

In [ ]:
lr_model = LogisticRegression(max_iter=5000, random_state=42)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

print(f"Exactitude régression logistique : {accuracy_score(y_test, y_pred_lr):.4f}")
print(classification_report(y_test, y_pred_lr))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, cmap="Blues")
plt.title("Régression logistique")
plt.show()

---
## 6. Forêt aléatoire

### Exercice 5.1 — Forêt aléatoire

Entraînez `RandomForestClassifier(n_estimators=100, random_state=42)` et comparez-la
à la régression logistique. Calculez aussi son exactitude **sur l'entraînement** :
que constatez-vous ?

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

print(f"Régression logistique : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Forêt aléatoire       : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Forêt, entraînement   : {accuracy_score(y_train, rf_model.predict(X_train_scaled)):.4f}")
print(classification_report(y_test, y_pred_rf))

**Réponse.** La forêt classe parfaitement les patients d'entraînement (1,00) mais fait moins bien
que la régression logistique sur le test : elle sur-apprend. Avec 54 patients de
test, une différence de quelques patients change l'exactitude de plusieurs points :
on y revient avec la validation croisée (partie 9).

---
## 7. Réseau de neurones (`MLPClassifier`)

Un **réseau de neurones** transforme l'entrée couche après couche. Chaque neurone
calcule une somme pondérée de ses entrées plus un biais, puis applique une
**fonction d'activation** non linéaire (ReLU par défaut). Les poids, comme les
coefficients θ de la régression logistique, sont **appris** par descente de gradient.

| | Régression logistique | Réseau de neurones (MLP) |
|---|---|---|
| **Couches** | une (entrée → sortie) | plusieurs (entrée → cachées → sortie) |
| **Frontière de décision** | linéaire | non linéaire |
| **Paramètres** | peu (nb de variables + 1) | beaucoup (selon l'architecture) |
| **Entraînement** | rapide, un seul minimum | plus lent, plusieurs minimums possibles |
| **Standardisation** | importante | **indispensable** |

Hyperparamètres principaux de `MLPClassifier` (*Multi-Layer Perceptron*) :

- `hidden_layer_sizes` : `(100,)` = une couche cachée de 100 neurones, `(64, 32)` = deux couches ;
- `activation` : `"relu"` (défaut), `"tanh"`, `"logistic"` ;
- `learning_rate_init` : taux d'apprentissage initial (0,001 par défaut) ;
- `max_iter` : nombre maximal d'époques (passages sur les données).

### Exercice 6.1 — Un premier réseau

Entraînez `MLPClassifier(max_iter=1000, random_state=42)` et comparez-le aux deux
modèles précédents.

In [ ]:
mlp_model = MLPClassifier(max_iter=1000, random_state=42)
mlp_model.fit(X_train_scaled, y_train)
y_pred_mlp = mlp_model.predict(X_test_scaled)

print(f"Régression logistique : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Forêt aléatoire       : {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Réseau de neurones    : {accuracy_score(y_test, y_pred_mlp):.4f}")
print(f"Réseau, entraînement  : {accuracy_score(y_train, mlp_model.predict(X_train_scaled)):.4f}")
print(classification_report(y_test, y_pred_mlp))

### Exercice 6.2 — La courbe de perte

Tracez la perte d'entraînement à chaque époque : elle dit si le réseau a convergé
(perte qui diminue puis se stabilise).

*Indice :* `plt.plot(mlp_model.loss_curve_)`, et `mlp_model.n_iter_` donne le nombre d'époques.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mlp_model.loss_curve_)
plt.xlabel("Époque")
plt.ylabel("Perte (entraînement)")
plt.title("Courbe de perte du MLP")
plt.show()
print(f"Perte finale : {mlp_model.loss_curve_[-1]:.4f} après {mlp_model.n_iter_} époques")

### Exercice 6.3 — L'arrêt précoce

Entraîné trop longtemps, un réseau sur-apprend : la perte d'entraînement continue de
baisser alors que le modèle se dégrade sur de nouveaux patients. **L'arrêt précoce**
met de côté une partie de l'entraînement comme validation et arrête l'apprentissage
quand le score de validation ne s'améliore plus :

- `early_stopping=True` active l'arrêt précoce ;
- `validation_fraction=0.1` : 10 % de l'entraînement sert de validation ;
- `n_iter_no_change=10` : arrêt après 10 époques sans amélioration.

Entraînez `mlp_es` avec l'arrêt précoce, comparez son nombre d'époques, son
exactitude et sa courbe de perte à celles de `mlp_model`. `mlp_es.validation_scores_`
contient l'exactitude de validation à chaque époque.

In [ ]:
mlp_es = MLPClassifier(max_iter=1000, random_state=42,
                       early_stopping=True, validation_fraction=0.1)
mlp_es.fit(X_train_scaled, y_train)
y_pred_es = mlp_es.predict(X_test_scaled)

print(f"Sans arrêt précoce : {mlp_model.n_iter_} époques, "
      f"test {accuracy_score(y_test, y_pred_mlp):.4f}, "
      f"rappel {recall_score(y_test, y_pred_mlp):.2f}")
print(f"Avec arrêt précoce : {mlp_es.n_iter_} époques, "
      f"test {accuracy_score(y_test, y_pred_es):.4f}, "
      f"rappel {recall_score(y_test, y_pred_es):.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(mlp_model.loss_curve_, label="sans arrêt précoce")
axes[0].plot(mlp_es.loss_curve_, label="avec arrêt précoce")
axes[0].set_xlabel("Époque"); axes[0].set_ylabel("Perte (entraînement)"); axes[0].legend()
axes[1].plot(mlp_es.validation_scores_)
axes[1].set_xlabel("Époque"); axes[1].set_ylabel("Exactitude de validation")
plt.tight_layout()
plt.show()

**Réponse.** Sans arrêt précoce, le réseau tourne des centaines d'époques et atteint une
exactitude d'entraînement proche de 1 : il apprend par cœur. Avec l'arrêt précoce il
s'arrête après quelques dizaines d'époques. Sur ces 54 patients de test l'exactitude
est voisine, mais le rappel (malades détectés) est meilleur. Attention : la
validation ne compte ici qu'une vingtaine de patients, donc le point d'arrêt est
lui-même bruité.

---
## 8. Régler le réseau

L'architecture (nombre et taille des couches cachées) change le résultat :

| Architecture | Description |
|---|---|
| `(50,)` | 1 couche, 50 neurones |
| `(100,)` | 1 couche, 100 neurones (défaut) |
| `(64, 32)` | 2 couches (64 → 32) |
| `(128, 64, 32)` | 3 couches : beaucoup de paramètres, il faut beaucoup de données |

**Règle pratique :** commencer simple, n'ajouter de la complexité que si elle aide.
Plus de couches ne veut pas dire meilleur, surtout avec 270 patients.

### Exercice 7.1 — Comparer des architectures

Pour chaque architecture, entraînez un MLP, affichez l'exactitude de test et tracez
les courbes de perte sur un même graphique.

In [ ]:
architectures = [(50,), (100,), (64, 32), (128, 64, 32)]
plt.figure(figsize=(10, 6))
for arch in architectures:
    model = MLPClassifier(hidden_layer_sizes=arch, max_iter=1000, random_state=42)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"{str(arch):<15} exactitude : {acc:.4f}  ({model.n_iter_} époques)")
    plt.plot(model.loss_curve_, label=f"{arch} (exactitude {acc:.3f})")
plt.xlabel("Époque")
plt.ylabel("Perte (entraînement)")
plt.title("Courbes de perte selon l'architecture")
plt.legend()
plt.show()

### Exercice 7.2 — Recherche sur grille

Avec `GridSearchCV` (validation croisée à 5 plis sur l'entraînement), cherchez la
meilleure combinaison de `hidden_layer_sizes`, `activation` et `learning_rate_init`,
puis évaluez le meilleur modèle sur le test.

```python
param_grid = {
    "hidden_layer_sizes": [(50,), (100,), (64, 32)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.01],
}
```

In [ ]:
param_grid = {
    "hidden_layer_sizes": [(50,), (100,), (64, 32)],
    "activation": ["relu", "tanh"],
    "learning_rate_init": [0.001, 0.01],
}
grid_search_mlp = GridSearchCV(MLPClassifier(max_iter=1000, random_state=42),
                               param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid_search_mlp.fit(X_train_scaled, y_train)

print("Meilleurs paramètres :", grid_search_mlp.best_params_)
print(f"Meilleure exactitude (validation croisée) : {grid_search_mlp.best_score_:.4f}")
y_pred_best_mlp = grid_search_mlp.best_estimator_.predict(X_test_scaled)
print(f"Exactitude de test : {accuracy_score(y_test, y_pred_best_mlp):.4f}")

---
## 9. Comparaison finale

### Exercice 8.1 — Comparaison par validation croisée

Un seul découpage de 54 patients de test est trop petit pour départager les modèles.
Comparez-les par validation croisée à 5 plis sur **toutes** les données. Pour que la
standardisation soit refaite dans chaque pli (sans fuite d'information du pli de
test), on la met dans un **pipeline** : `make_pipeline(StandardScaler(), modèle)`.

In [ ]:
models = {
    "Régression logistique": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=42)),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=100, random_state=42),
    "Réseau de neurones": make_pipeline(StandardScaler(), grid_search_mlp.best_estimator_),
}
print(f"{'Modèle':<25} {'Exactitude':>11} {'Écart type':>11}")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    print(f"{name:<25} {scores.mean():>11.4f} {scores.std():>11.4f}")

### Exercice 8.2 — Matrices de confusion côte à côte

Affichez les matrices de confusion des trois modèles sur le jeu de test.

*Indice :* `fig, axes = plt.subplots(1, 3, figsize=(18, 5))`

**Question :** pour dépister une maladie cardiaque, quelle case de la matrice
regardez-vous en premier ?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
predictions = {
    "Régression logistique": y_pred_lr,
    "Forêt aléatoire": y_pred_rf,
    "Réseau de neurones": y_pred_best_mlp,
}
for ax, (name, y_pred) in zip(axes, predictions.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap="Blues",
                                            colorbar=False)
    ax.set_title(f"{name}\nrappel {recall_score(y_test, y_pred):.2f}")
plt.tight_layout()
plt.show()

**Réponse.** La case en bas à gauche : les **faux négatifs**, des malades prédits sains. Un
dépistage qui les manque est dangereux ; on regarde donc le **rappel** de la classe
« maladie présente » autant que l'exactitude.

---
## 10. Bilan des trois séances

| | Régression logistique | Forêt aléatoire | Réseau de neurones (MLP) |
|---|---|---|---|
| **Type** | linéaire | ensemble d'arbres | couches de neurones |
| **Frontière** | linéaire | non linéaire | non linéaire |
| **Standardisation** | importante | inutile | indispensable |
| **Interprétabilité** | forte (coefficients) | moyenne (importance des variables) | faible (boîte noire) |
| **Quand l'utiliser** | référence, petits jeux | données tabulaires, défaut robuste | motifs complexes, beaucoup de données |

**La démarche (pour tous les modèles)** : explorer, visualiser, préparer, découper,
entraîner (en commençant par une référence), évaluer avec la bonne métrique,
régler par validation croisée, comparer.

> En santé, **le choix de la métrique compte plus que le choix du modèle**. Un
> modèle exact à 95 % mais qui manque des malades est dangereux : un diagnostic
> manqué (faux négatif) coûte en général bien plus qu'une fausse alerte.